In [4]:
"""
Enhanced CoolProp Calculator
A comprehensive tool for thermophysical property calculations with multiple interfaces
"""

import CoolProp.CoolProp as cp
import json
import csv
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
import sys


class CoolPropCalculator:
    """Enhanced CoolProp calculator with multiple calculation modes"""
    
    # Property definitions
    PROPERTIES = {
        'T': {'name': 'Temperature', 'unit': 'K', 'description': 'Absolute temperature'},
        'P': {'name': 'Pressure', 'unit': 'Pa', 'description': 'Absolute pressure'},
        'H': {'name': 'Enthalpy', 'unit': 'J/kg', 'description': 'Specific enthalpy'},
        'S': {'name': 'Entropy', 'unit': 'J/kg·K', 'description': 'Specific entropy'},
        'D': {'name': 'Density', 'unit': 'kg/m³', 'description': 'Mass density'},
        'Q': {'name': 'Quality', 'unit': '-', 'description': 'Vapor quality (0=liquid, 1=vapor)'},
        'V': {'name': 'Specific Volume', 'unit': 'm³/kg', 'description': 'Specific volume'},
        'U': {'name': 'Internal Energy', 'unit': 'J/kg', 'description': 'Specific internal energy'},
        'G': {'name': 'Gibbs Energy', 'unit': 'J/kg', 'description': 'Specific Gibbs free energy'},
        'A': {'name': 'Helmholtz Energy', 'unit': 'J/kg', 'description': 'Specific Helmholtz free energy'},
        'viscosity': {'name': 'Viscosity', 'unit': 'Pa·s', 'description': 'Dynamic viscosity'},
        'conductivity': {'name': 'Thermal Conductivity', 'unit': 'W/m·K', 'description': 'Thermal conductivity'},
        'Prandtl': {'name': 'Prandtl Number', 'unit': '-', 'description': 'Prandtl number'},
        'surface_tension': {'name': 'Surface Tension', 'unit': 'N/m', 'description': 'Surface tension'},
        'speed_sound': {'name': 'Speed of Sound', 'unit': 'm/s', 'description': 'Speed of sound'},
        'C': {'name': 'Specific Heat (const P)', 'unit': 'J/kg·K', 'description': 'Specific heat at constant pressure'},
        'CVMASS': {'name': 'Specific Heat (const V)', 'unit': 'J/kg·K', 'description': 'Specific heat at constant volume'},
        'isentropic_expansion_coefficient': {'name': 'Isentropic Exp. Coeff.', 'unit': '-', 'description': 'Isentropic expansion coefficient'},
        'isothermal_compressibility': {'name': 'Isothermal Compress.', 'unit': '1/Pa', 'description': 'Isothermal compressibility'},
    }
    
    # Critical properties
    CRITICAL_PROPERTIES = {
        'T_critical': {'name': 'Critical Temperature', 'unit': 'K'},
        'P_critical': {'name': 'Critical Pressure', 'unit': 'Pa'},
        'rhocrit': {'name': 'Critical Density', 'unit': 'kg/m³'},
    }
    
    # Triple point properties
    TRIPLE_PROPERTIES = {
        'T_triple': {'name': 'Triple Point Temperature', 'unit': 'K'},
        'p_triple': {'name': 'Triple Point Pressure', 'unit': 'Pa'},
    }
    
    # Common substances (expanded list)
    COMMON_SUBSTANCES = [
        'Water', 'Air', 'Nitrogen', 'Oxygen', 'Hydrogen', 'Helium',
        'Ammonia', 'CarbonDioxide', 'Methane', 'Ethane', 'Propane', 'n-Butane',
        'R134a', 'R410A', 'R404A', 'R407C', 'R32', 'R1234yf',
        'Argon', 'Ethanol', 'Methanol', 'Toluene', 'Benzene',
    ]
    
    def __init__(self):
        self.calculation_history = []
    
    def calculate_property(self, output_prop: str, input1_prop: str, input1_value: float,
                          input2_prop: str, input2_value: float, substance: str) -> Dict[str, Any]:
        """
        Calculate a thermophysical property
        
        Returns:
            Dictionary with result, metadata, and status
        """
        try:
            result = cp.PropsSI(output_prop, input1_prop, input1_value, 
                              input2_prop, input2_value, substance)
            
            calculation = {
                'timestamp': datetime.now().isoformat(),
                'substance': substance,
                'output_property': output_prop,
                'output_value': result,
                'input1_property': input1_prop,
                'input1_value': input1_value,
                'input2_property': input2_prop,
                'input2_value': input2_value,
                'status': 'success'
            }
            
            self.calculation_history.append(calculation)
            return calculation
            
        except Exception as e:
            error_calc = {
                'timestamp': datetime.now().isoformat(),
                'substance': substance,
                'output_property': output_prop,
                'error': str(e),
                'status': 'error'
            }
            self.calculation_history.append(error_calc)
            return error_calc
    
    def get_critical_properties(self, substance: str) -> Dict[str, float]:
        """Get all critical properties for a substance"""
        try:
            return {
                'T_critical': cp.PropsSI('T_critical', substance),
                'P_critical': cp.PropsSI('P_critical', substance),
                'rhocrit': cp.PropsSI('rhocrit', substance),
            }
        except Exception as e:
            return {'error': str(e)}
    
    def get_triple_properties(self, substance: str) -> Dict[str, float]:
        """Get triple point properties for a substance"""
        try:
            return {
                'T_triple': cp.PropsSI('T_triple', substance),
                'p_triple': cp.PropsSI('p_triple', substance),
            }
        except Exception as e:
            return {'error': str(e)}
    
    def get_all_properties(self, input1_prop: str, input1_value: float,
                          input2_prop: str, input2_value: float, 
                          substance: str) -> Dict[str, Any]:
        """Calculate all available properties at once"""
        results = {
            'substance': substance,
            'inputs': {
                input1_prop: input1_value,
                input2_prop: input2_value
            },
            'properties': {}
        }
        
        for prop_key in self.PROPERTIES.keys():
            if prop_key not in [input1_prop, input2_prop]:
                try:
                    value = cp.PropsSI(prop_key, input1_prop, input1_value,
                                     input2_prop, input2_value, substance)
                    results['properties'][prop_key] = {
                        'value': value,
                        'unit': self.PROPERTIES[prop_key]['unit'],
                        'name': self.PROPERTIES[prop_key]['name']
                    }
                except:
                    pass
        
        return results
    
    def batch_calculate(self, calculations: List[Dict]) -> List[Dict]:
        """
        Process multiple calculations at once
        
        Args:
            calculations: List of calculation dictionaries with keys:
                         output_prop, input1_prop, input1_value, input2_prop, input2_value, substance
        """
        results = []
        for calc in calculations:
            result = self.calculate_property(**calc)
            results.append(result)
        return results
    
    def export_history(self, filename: str, format: str = 'json'):
        """Export calculation history to file"""
        if format == 'json':
            with open(filename, 'w') as f:
                json.dump(self.calculation_history, f, indent=2)
        elif format == 'csv':
            if self.calculation_history:
                with open(filename, 'w', newline='') as f:
                    writer = csv.DictWriter(f, fieldnames=self.calculation_history[0].keys())
                    writer.writeheader()
                    writer.writerows(self.calculation_history)
        print(f"History exported to {filename}")
    
    def create_state_point_table(self, substance: str, temps: List[float], 
                                pressures: List[float]) -> List[Dict]:
        """Create a table of properties at different state points"""
        table = []
        for T in temps:
            for P in pressures:
                try:
                    state = {
                        'T': T,
                        'P': P,
                        'H': cp.PropsSI('H', 'T', T, 'P', P, substance),
                        'S': cp.PropsSI('S', 'T', T, 'P', P, substance),
                        'D': cp.PropsSI('D', 'T', T, 'P', P, substance),
                        'viscosity': cp.PropsSI('V', 'T', T, 'P', P, substance),
                        'conductivity': cp.PropsSI('L', 'T', T, 'P', P, substance),
                    }
                    table.append(state)
                except:
                    pass
        return table


class InteractiveCLI:
    """Enhanced interactive command-line interface"""
    
    def __init__(self):
        self.calc = CoolPropCalculator()
        self.running = True
    
    def display_menu(self):
        """Display main menu"""
        print("\n" + "="*60)
        print("  ENHANCED COOLPROP CALCULATOR")
        print("="*60)
        print("\n[1] Single Property Calculation")
        print("[2] All Properties at State Point")
        print("[3] Critical Properties")
        print("[4] Triple Point Properties")
        print("[5] Batch Calculations")
        print("[6] Property Table Generator")
        print("[7] View Calculation History")
        print("[8] Export History")
        print("[9] List Available Substances")
        print("[10] Property Reference Guide")
        print("[0] Exit")
        print("\n" + "-"*60)
    
    def single_calculation(self):
        """Interactive single property calculation"""
        print("\n--- Single Property Calculation ---\n")
        
        # Show available properties
        print("Available properties:")
        for i, (key, val) in enumerate(self.calc.PROPERTIES.items(), 1):
            print(f"  {key:15} - {val['name']} [{val['unit']}]")
        
        output_prop = input("\nProperty to calculate: ").strip()
        if output_prop not in self.calc.PROPERTIES:
            print("Invalid property!")
            return
        
        substance = input("Substance: ").strip()
        
        print("\nFirst input property:")
        input1_prop = input("Property: ").strip()
        if input1_prop not in self.calc.PROPERTIES:
            print("Invalid property!")
            return
        input1_value = float(input(f"Value [{self.calc.PROPERTIES[input1_prop]['unit']}]: "))
        
        print("\nSecond input property:")
        input2_prop = input("Property: ").strip()
        if input2_prop not in self.calc.PROPERTIES:
            print("Invalid property!")
            return
        input2_value = float(input(f"Value [{self.calc.PROPERTIES[input2_prop]['unit']}]: "))
        
        result = self.calc.calculate_property(output_prop, input1_prop, input1_value,
                                             input2_prop, input2_value, substance)
        
        if result['status'] == 'success':
            print(f"\n{'='*60}")
            print(f"Result: {self.calc.PROPERTIES[output_prop]['name']}")
            print(f"Value:  {result['output_value']:.6e} {self.calc.PROPERTIES[output_prop]['unit']}")
            print(f"{'='*60}")
        else:
            print(f"\nError: {result['error']}")
    
    def all_properties(self):
        """Calculate all properties at a state point"""
        print("\n--- All Properties at State Point ---\n")
        
        substance = input("Substance: ").strip()
        
        print("\nFirst input property:")
        input1_prop = input("Property (e.g., T, P): ").strip()
        input1_value = float(input(f"Value [{self.calc.PROPERTIES.get(input1_prop, {'unit': '?'})['unit']}]: "))
        
        print("\nSecond input property:")
        input2_prop = input("Property (e.g., P, D): ").strip()
        input2_value = float(input(f"Value [{self.calc.PROPERTIES.get(input2_prop, {'unit': '?'})['unit']}]: "))
        
        results = self.calc.get_all_properties(input1_prop, input1_value,
                                              input2_prop, input2_value, substance)
        
        print(f"\n{'='*70}")
        print(f"All Properties for {substance}")
        print(f"At {input1_prop}={input1_value}, {input2_prop}={input2_value}")
        print(f"{'='*70}\n")
        
        for prop, data in results['properties'].items():
            print(f"{data['name']:30} = {data['value']:15.6e} {data['unit']}")
    
    def critical_properties(self):
        """Display critical properties"""
        print("\n--- Critical Properties ---\n")
        substance = input("Substance: ").strip()
        
        props = self.calc.get_critical_properties(substance)
        
        if 'error' not in props:
            print(f"\nCritical Properties for {substance}:")
            print(f"  Temperature: {props['T_critical']:.2f} K")
            print(f"  Pressure:    {props['P_critical']:.2e} Pa")
            print(f"  Density:     {props['rhocrit']:.2f} kg/m³")
        else:
            print(f"Error: {props['error']}")
    
    def list_substances(self):
        """List common substances"""
        print("\n--- Common Substances ---\n")
        for i, substance in enumerate(self.calc.COMMON_SUBSTANCES, 1):
            print(f"{i:3}. {substance}")
        print("\nNote: Many more substances are available. See CoolProp documentation.")
    
    def property_guide(self):
        """Display property reference guide"""
        print("\n" + "="*70)
        print("PROPERTY REFERENCE GUIDE")
        print("="*70 + "\n")
        
        for key, val in self.calc.PROPERTIES.items():
            print(f"{key:15} | {val['name']:25} | {val['unit']:12} | {val['description']}")
    
    def view_history(self):
        """View calculation history"""
        print("\n--- Calculation History ---\n")
        if not self.calc.calculation_history:
            print("No calculations in history.")
            return
        
        for i, calc in enumerate(self.calc.calculation_history[-10:], 1):
            if calc['status'] == 'success':
                print(f"{i}. {calc['substance']}: {calc['output_property']} = {calc['output_value']:.4e}")
            else:
                print(f"{i}. Error: {calc.get('error', 'Unknown error')}")
    
    def export_history_menu(self):
        """Export history menu"""
        print("\n--- Export History ---\n")
        filename = input("Filename: ").strip()
        format_choice = input("Format (json/csv) [json]: ").strip() or 'json'
        
        self.calc.export_history(filename, format_choice)
    
    def run(self):
        """Run the interactive CLI"""
        while self.running:
            self.display_menu()
            choice = input("\nSelect option: ").strip()
            
            try:
                if choice == '1':
                    self.single_calculation()
                elif choice == '2':
                    self.all_properties()
                elif choice == '3':
                    self.critical_properties()
                elif choice == '4':
                    substance = input("\nSubstance: ").strip()
                    props = self.calc.get_triple_properties(substance)
                    if 'error' not in props:
                        print(f"\nTriple Point for {substance}:")
                        print(f"  Temperature: {props['T_triple']:.2f} K")
                        print(f"  Pressure:    {props['p_triple']:.2e} Pa")
                    else:
                        print(f"Error: {props['error']}")
                elif choice == '7':
                    self.view_history()
                elif choice == '8':
                    self.export_history_menu()
                elif choice == '9':
                    self.list_substances()
                elif choice == '10':
                    self.property_guide()
                elif choice == '0':
                    print("\nGoodbye!")
                    self.running = False
                else:
                    print("Invalid option!")
            except KeyboardInterrupt:
                print("\n\nOperation cancelled.")
            except Exception as e:
                print(f"\nError: {e}")


def main():
    """Main entry point"""
    if len(sys.argv) > 1 and sys.argv[1] == '--simple':
        # Simple mode (original style)
        calc = CoolPropCalculator()
        print("\nCOOLPROP CALCULATOR (Simple Mode)\n")
        # ... original simple mode code ...
    else:
        # Enhanced interactive mode
        cli = InteractiveCLI()
        cli.run()


if __name__ == "__main__":
    main()


  ENHANCED COOLPROP CALCULATOR

[1] Single Property Calculation
[2] All Properties at State Point
[3] Critical Properties
[4] Triple Point Properties
[5] Batch Calculations
[6] Property Table Generator
[7] View Calculation History
[8] Export History
[9] List Available Substances
[10] Property Reference Guide
[0] Exit

------------------------------------------------------------

Select option: 2

--- All Properties at State Point ---

Substance: Water

First input property:
Property (e.g., T, P): T
Value [K]: 298.15

Second input property:
Property (e.g., P, D): P
Value [Pa]: 101325

All Properties for Water
At T=298.15, P=101325.0

Enthalpy                       =    1.049201e+05 J/kg
Entropy                        =    3.671996e+02 J/kg·K
Density                        =    9.970476e+02 kg/m³
Quality                        =   -1.000000e+00 -
Specific Volume                =    8.900225e-04 m³/kg
Internal Energy                =    1.048185e+05 J/kg
Gibbs Energy                   = 